In [1]:
import pdfplumber, re
import pandas as pd
from transformers import pipeline
import geemap
import folium

In [2]:
clf = pipeline("text-classification", model="climatebert/environmental-claims")
specificity_clf = pipeline("text-classification", model="climatebert/distilroberta-base-climate-specificity")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
def extract_claims(pdf_path, company_name):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + " "

    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

    claims = []
    for s in sentences:
        result = clf(s, truncation=True, max_length=512)[0]
        if result["label"] == "yes":
            claims.append(s)

    df = pd.DataFrame({"company": company_name, "claim": claims})
    df.to_csv(f"data/claims/{company_name}_claims.csv", index=False)
    print(f"{company_name}: {len(sentences)} sentences, {len(claims)} claims")
    return df

In [4]:
# Change these two lines only when switching to a new company/report
pdf_path = "Reports/sdguthrie.pdf"
company_name = "sdguthrie"

In [5]:
sdg_df = extract_claims(pdf_path, company_name)
sdg_df

sdguthrie: 882 sentences, 122 claims


,company,claim
0,sdguthrie,"By harnessing the power of innovation,\nwe unl..."
1,sdguthrie,Our vision of a “Beyond Zero” future inspires ...
2,sdguthrie,"Launched in 2024, our “Beyond Zero” developmen..."
3,sdguthrie,"Solomon Islands, with 35,799 benefitting from ..."
4,sdguthrie,GOING BEYOND\nAs we scale our business and sus...
...,...,...
117,sdguthrie,NBPOL also supported\nIndonesia’s palm oil ind...
118,sdguthrie,MOVING FORWARD\nIndonesia\nProgramme Number of...
119,sdguthrie,"By prioritising customer privacy, data protect..."
120,sdguthrie,upholding the highest ethical standards across...


In [6]:
def tag_claim(text):
    t = text.lower()
    specificity_result = specificity_clf(text, truncation=True, max_length=512)[0]
    return {
        "claim": text,
        "deforestation": "deforestation" in t,
        "traceability": "traceab" in t or "traceable" in t,
        "ndpe": "ndpe" in t or "no deforestation" in t,
        "peat": "peat" in t,
        "deadline": next((y for y in ["2025","2030","2050","2013"] if y in t), None),
        "strength": "hard" if ("no deforestation" in t or "deforestation-free" in t or "zero" in t) else "soft",
        "specificity": specificity_result["label"],
        "specificity_score": specificity_result["score"],
    }

tagged = pd.DataFrame([tag_claim(c) for c in sdg_df ["claim"]])
tagged.to_csv(f"data/claims/{company_name}_tagged.csv", index=False)
tagged

,claim,deforestation,traceability,ndpe,peat,deadline,strength,specificity,specificity_score
0,"By harnessing the power of innovation,\nwe unl...",False,False,False,False,NaN,soft,non,0.781647
1,Our vision of a “Beyond Zero” future inspires ...,False,False,False,False,NaN,hard,non,0.850353
2,"Launched in 2024, our “Beyond Zero” developmen...",False,False,False,False,NaN,hard,non,0.608760
3,"Solomon Islands, with 35,799 benefitting from ...",False,False,False,False,NaN,soft,spec,0.802242
4,GOING BEYOND\nAs we scale our business and sus...,False,False,False,False,NaN,hard,spec,0.779192
...,...,...,...,...,...,...,...,...,...
117,NBPOL also supported\nIndonesia’s palm oil ind...,False,False,False,False,NaN,soft,spec,0.614121
118,MOVING FORWARD\nIndonesia\nProgramme Number of...,False,False,False,False,NaN,hard,spec,0.705242
119,"By prioritising customer privacy, data protect...",False,False,False,False,NaN,soft,non,0.913277
120,upholding the highest ethical standards across...,False,False,False,False,NaN,soft,non,0.793104


In [7]:
specific_claims = tagged[tagged["specificity"] == "spec"].reset_index(drop=True)
specific_claims.to_csv(f"data/claims/{company_name}_specific_claims.csv", index=False)
print(f"{len(specific_claims)} of {len(tagged)} claims are specific (checkable)")
specific_claims

50 of 122 claims are specific (checkable)


,claim,deforestation,traceability,ndpe,peat,deadline,strength,specificity,specificity_score
0,"Solomon Islands, with 35,799 benefitting from ...",False,False,False,False,NaN,soft,spec,0.802242
1,GOING BEYOND\nAs we scale our business and sus...,False,False,False,False,NaN,hard,spec,0.779192
2,Leveraging our supply chain in Papua New Guine...,False,False,False,False,2030,soft,spec,0.893180
3,We are developing a regenerative agricultural\...,False,False,False,False,NaN,soft,spec,0.874616
4,Transforming Lives Through • Community Rights ...,True,True,False,False,NaN,hard,spec,0.671456
5,through capacity building and\nTackling Waste ...,False,False,False,False,NaN,soft,spec,0.552657
6,"Restore, rehabilitate or conserve – Reforestat...",False,False,False,False,2030,hard,spec,0.709963
7,100% 100%\nAchieved Achieved Became the\nMalay...,False,False,False,False,NaN,hard,spec,0.802048
8,We are guided by approved by SBTi\nZero 2030 t...,False,False,False,False,2030,hard,spec,0.665755
9,Achieved for greenhouse gas\ntraceability to m...,True,True,True,True,NaN,hard,spec,0.831286


In [8]:
gar_final = tagged[
    (tagged["specificity"] == "spec") &
    (tagged["deforestation"] | tagged["ndpe"] | tagged["peat"] | tagged["traceability"])
]
gar_final.to_csv(f"data/claims/{company_name}_final_claims.csv", index=False)
gar_final

,claim,deforestation,traceability,ndpe,peat,deadline,strength,specificity,specificity_score
9,Transforming Lives Through • Community Rights ...,True,True,False,False,NaN,hard,spec,0.671456
18,Achieved for greenhouse gas\ntraceability to m...,True,True,True,True,NaN,hard,spec,0.831286
23,"We extend our efforts beyond our operations, a...",False,False,False,True,2030,soft,spec,0.669504
45,With 67% of our net emissions reduced our Scop...,False,False,True,True,NaN,soft,spec,0.892418
49,and compliance with our deforestation-free com...,True,False,False,False,NaN,hard,spec,0.512257
53,significant milestone by delivering its first ...,True,False,True,False,NaN,hard,spec,0.962195
54,"of 24,250 MT arrived at SD Guthrie Internation...",True,True,False,False,NaN,hard,spec,0.946000
65,Traceability to Plantation\nDeforestation-Free...,True,True,False,False,NaN,hard,spec,0.595711
67,This pilot shipment is fully traceable and ver...,True,True,False,False,NaN,hard,spec,0.587408
69,by enabling traceability of raw materials thro...,True,True,True,True,NaN,hard,spec,0.559919


In [9]:
uml_path = "UML-Jan-2026.csv"
uml = pd.read_csv(uml_path, sep=";", encoding="latin1")
uml.columns = [" ".join(str(column).replace("﻿", "").split()) for column in uml.columns]

group_col = next((column for column in uml.columns if column.casefold() == "group name"), None)
if group_col is None:
    raise KeyError(f"'Group Name' column not found. Loaded columns: {list(uml.columns)}")

sdg_mills = uml[
    uml[group_col].str.contains("SIME DARBY", case=False, na=False, regex=False)
].reset_index(drop=True)

print("SD Guthrie mills:", len(sdg_mills))
sdg_mills

SD Guthrie mills: 2


,UML ID,Group Name,Parent Company,Mill Name,RSPO Status,RSPO type,Date RSPO Certification Status,Latitude,Longitude,GPS coordinates,ISO,Country,Province,District,Confidence level,Alternative name
0,PO1000010727,SIME DARBY PLANTATION BHD,MARKHAM FARMING COMPANY LTD,ERAP / MARKHAM FAMRING,RSPO Certified,"RSPO Certified, IP, MB",13/01/2026,-658.087,14.664.222,"-6.58087, 146.64222",PNG,Papua New Guinea,Morobe,Chivasing,2-High Confidence,NaN
1,PO1000011310,SIME DARBY,PT PADANG PALMA PERMAI,TAMIANG POM,Not RSPO Certified,NaN,13/01/2026,430.555,980.267,"4.30555, 98.0267",IDN,Indonesia,Aceh,Aceh Tamiang,2-High Confidence,NaN


In [10]:
for name in ["SIME DARBY", "GUTHRIE", "SD GUTHRIE", "SIME"]:
    n = uml[group_col].str.contains(name, case=False, na=False, regex=False).sum()
    print(f"'{name}': {n} in Group Name")
    # Parent Company માં પણ જુઓ
    pc = uml["Parent Company"].str.contains(name, case=False, na=False, regex=False).sum()
    print(f"'{name}': {pc} in Parent Company")

'SIME DARBY': 2 in Group Name
'SIME DARBY': 39 in Parent Company
'GUTHRIE': 0 in Group Name
'GUTHRIE': 1 in Parent Company
'SD GUTHRIE': 0 in Group Name
'SD GUTHRIE': 0 in Parent Company
'SIME': 66 in Group Name
'SIME': 40 in Parent Company


In [11]:
# "Latitude"/"Longitude" columns are corrupted (extra thousand-separator dots),
# so parse the clean "lat, lon" pairs out of "GPS coordinates" instead.
_coords = sdg_mills["GPS coordinates"].str.split(",", expand=True)

mill_coords = sdg_mills[[group_col, "Mill Name"]].copy()
mill_coords["latitude"] = pd.to_numeric(_coords[0].str.strip(), errors="coerce")
mill_coords["longitude"] = pd.to_numeric(_coords[1].str.strip(), errors="coerce")
mill_coords = mill_coords.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

mill_coords

,Group Name,Mill Name,latitude,longitude
0,SIME DARBY PLANTATION BHD,ERAP / MARKHAM FAMRING,-6.58087,146.64222
1,SIME DARBY,TAMIANG POM,4.30555,98.02670


In [12]:
import ee
ee.Authenticate()

True

In [13]:
ee.Initialize(project='thesis-greenwashing')
print("Connected!")

Connected!


---
# Layer 2a — Multi-company forest loss
Reusable function computing forest cover, total loss, and post-2020 loss in a single Earth Engine call per mill (Hansen v1.13, 2000-2025). Works for any company via keyword match against the UML group name.

In [14]:
# Reusable mill extractor — searches BOTH Group Name and Parent Company.
# Many companies (e.g. SD Guthrie / SIME DARBY) have their mills listed under
# Parent Company, not Group Name — searching only one column undercounts them.
# Coordinates are parsed from "GPS coordinates" because the Latitude/Longitude
# columns in the UML are corrupted (thousand-separator dots).

def get_company_mills(uml_df, keyword, company_name):
    mask = (
        uml_df[group_col].str.contains(keyword, case=False, na=False, regex=True)
        | uml_df["Parent Company"].str.contains(keyword, case=False, na=False, regex=True)
    )
    mills = uml_df[mask].drop_duplicates(subset=["Mill Name", "GPS coordinates"]).reset_index(drop=True)

    coords = mills["GPS coordinates"].str.split(",", expand=True)

    out = mills[[group_col, "Mill Name", "Country"]].copy()
    out["company"] = company_name
    out["latitude"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
    out["longitude"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")
    out = out.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

    print(f"{company_name}: {len(out)} mills with valid coordinates")
    return out

In [15]:
# One Earth Engine call per mill — returns forest 2000, total loss, post-2020 loss
# Uses Hansen v1.13 (2000-2025) throughout. lossyear >= 21 means 2021 onwards = EUDR window.

hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
pixel_ha = ee.Image.pixelArea().divide(10000)

_bands = (
    hansen.select("treecover2000").gt(30).multiply(pixel_ha).rename("forest_2000_ha")
    .addBands(hansen.select("lossyear").gt(0).multiply(pixel_ha).rename("loss_total_ha"))
    .addBands(hansen.select("lossyear").gte(21).multiply(pixel_ha).rename("loss_post2020_ha"))
)

def mill_forest_loss(lon, lat, buffer_m=10000):
    region = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    stats = _bands.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=30,
        maxPixels=1e10,
    ).getInfo()
    return (
        stats.get("forest_2000_ha") or 0,
        stats.get("loss_total_ha") or 0,
        stats.get("loss_post2020_ha") or 0,
    )

In [16]:
import time
def company_forest_loss(uml_df, keyword, company_name, buffer_m=10000):
    mills = get_company_mills(uml_df, keyword, company_name)
    rows = []
    for i, (_, m) in enumerate(mills.iterrows(), 1):
        try:
            f, lt, lp = mill_forest_loss(m["longitude"], m["latitude"], buffer_m)
            rows.append({
                "company": company_name,
                "mill_name": m["Mill Name"],
                "country": m["Country"],
                "latitude": m["latitude"],
                "longitude": m["longitude"],
                "forest_2000_ha": f,
                "loss_total_ha": lt,
                "loss_post2020_ha": lp,
            })
            print(f"  [{i}/{len(mills)}] {m['Mill Name']}: {lp:,.0f} ha post-2020")
        except Exception as exc:
            print(f"  [{i}/{len(mills)}] FAILED {m['Mill Name']}: {exc}")
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    df.to_csv(f"data/mills/{company_name}_mills.csv", index=False)
    return df

## Verification step

Run Wilmar first and check the total against the figure already on record.
If it reproduces, the results are yours and defensible. If it does not,
find out why now — not in the viva.

This may take 5-15 minutes depending on how many mills Wilmar has.
Progress prints after each mill.

In [17]:
wilmar_mills = company_forest_loss(uml, "WILMAR", "wilmar")

print("\n=== WILMAR TOTAL ===")
print(f"Mills processed: {len(wilmar_mills)}")
print(f"Forest 2000:     {wilmar_mills['forest_2000_ha'].sum():,.0f} ha")
print(f"Loss total:      {wilmar_mills['loss_total_ha'].sum():,.0f} ha")
print(f"Loss post-2020:  {wilmar_mills['loss_post2020_ha'].sum():,.0f} ha")

wilmar: 45 mills with valid coordinates
  [1/45] SABAHMAS: 6,675 ha post-2020
  [2/45] SAREMAS 1: 3,219 ha post-2020
  [3/45] SAREMAS 2: 4,849 ha post-2020
  [4/45] TERUSAN: 1,352 ha post-2020
  [5/45] SAPI: 2,049 ha post-2020
  [6/45] REKA HALUS: 2,394 ha post-2020
  [7/45] MUSTIKA SEMBULUH 1: 2,237 ha post-2020
  [8/45] KERRY SAWIT INDONESIA 1: 511 ha post-2020
  [9/45] RIBUBONUS: 1,309 ha post-2020
  [10/45] SRI KAMUSAN: 3,019 ha post-2020
  [11/45] PINANG AWAM: 3,978 ha post-2020
  [12/45] KENCANA SAWIT INDONESIA: 9,097 ha post-2020
  [13/45] BURNAI TIMUR: 2,829 ha post-2020
  [14/45] SARANA TITIAN PERMATA PKS 1: 497 ha post-2020
  [15/45] AMP PLANTATION: 1,720 ha post-2020
  [16/45] DABUK REJO: 3,856 ha post-2020
  [17/45] BUMI SAWIT KENCANA: 1,023 ha post-2020
  [18/45] BENSO OIL PALM PLANTATION OIL MILL: 4,057 ha post-2020
  [19/45] DAYA LABUHAN INDAH 2: 3,267 ha post-2020
  [20/45] GERSINDO MINANG PLANTATION: 2,994 ha post-2020
  [21/45] MENTAYA SAWIT MAS: 1,237 ha post-2020
  

## Run all companies

Only run this once Wilmar has verified. Keyword patterns include former
names and parent groups — SD Guthrie is listed as SIME DARBY in the UML.

Check the mill counts printed for each company. If any company returns
0 mills, its keyword needs adjusting before you trust the master table.

In [18]:
COMPANIES = {
    "wilmar":          "WILMAR",
    "gar":             "GOLDEN AGRI|SINAR MAS",
    "sdguthrie":       "SIME DARBY|GUTHRIE",
    "musimmas":        "MUSIM MAS",
    "ioi":             "IOI CORPORATION|IOI GROUP",
    "klk":             "KUALA LUMPUR KEPONG|KLK",
    "bumitama":        "BUMITAMA",
    "astraagro":       "ASTRA AGRO|ASTRA",
    "genting":         "GENTING",
    "sipef":           "SIPEF",
    "firstresources":  "FIRST RESOURCES",
}
all_mills = []
for _name, _keyword in COMPANIES.items():
    print(f"\n=== {_name.upper()} ===")
    all_mills.append(company_forest_loss(uml, _keyword, _name))
mills_master = pd.concat(all_mills, ignore_index=True)
mills_master.to_csv("data/mills/all_mills_forest_loss.csv", index=False)
print(f"\nSaved {len(mills_master)} mills across {len(COMPANIES)} companies")


=== WILMAR ===
wilmar: 45 mills with valid coordinates
  [1/45] SABAHMAS: 6,675 ha post-2020
  [2/45] SAREMAS 1: 3,219 ha post-2020
  [3/45] SAREMAS 2: 4,849 ha post-2020
  [4/45] TERUSAN: 1,352 ha post-2020
  [5/45] SAPI: 2,049 ha post-2020
  [6/45] REKA HALUS: 2,394 ha post-2020
  [7/45] MUSTIKA SEMBULUH 1: 2,237 ha post-2020
  [8/45] KERRY SAWIT INDONESIA 1: 511 ha post-2020
  [9/45] RIBUBONUS: 1,309 ha post-2020
  [10/45] SRI KAMUSAN: 3,019 ha post-2020
  [11/45] PINANG AWAM: 3,978 ha post-2020
  [12/45] KENCANA SAWIT INDONESIA: 9,097 ha post-2020
  [13/45] BURNAI TIMUR: 2,829 ha post-2020
  [14/45] SARANA TITIAN PERMATA PKS 1: 497 ha post-2020
  [15/45] AMP PLANTATION: 1,720 ha post-2020
  [16/45] DABUK REJO: 3,856 ha post-2020
  [17/45] BUMI SAWIT KENCANA: 1,023 ha post-2020
  [18/45] BENSO OIL PALM PLANTATION OIL MILL: 4,057 ha post-2020
  [19/45] DAYA LABUHAN INDAH 2: 3,267 ha post-2020
  [20/45] GERSINDO MINANG PLANTATION: 2,994 ha post-2020
  [21/45] MENTAYA SAWIT MAS: 1,237

### Check mill counts first

Confirm each company matches enough mills across both columns before running the full loop. Fix any keyword that comes back far too low.

In [19]:
# --- Verify mill counts across BOTH columns before trusting the master table ---
# Run this after defining COMPANIES. Any company far below its known mill count
# has a keyword problem and must be fixed before its forest-loss figure is used.

for _name, _kw in COMPANIES.items():
    _g = uml[group_col].str.contains(_kw, case=False, na=False, regex=True).sum()
    _p = uml["Parent Company"].str.contains(_kw, case=False, na=False, regex=True).sum()
    _both = (
        uml[group_col].str.contains(_kw, case=False, na=False, regex=True)
        | uml["Parent Company"].str.contains(_kw, case=False, na=False, regex=True)
    ).sum()
    print(f"{_name:16s} group={_g:3d}  parent={_p:3d}  combined={_both:3d}")

wilmar           group= 45  parent=  1  combined= 45
gar              group= 48  parent=  6  combined= 50
sdguthrie        group=  2  parent= 40  combined= 42
musimmas         group= 17  parent=  4  combined= 19
ioi              group= 15  parent= 10  combined= 15
klk              group= 29  parent= 10  combined= 30
bumitama         group= 14  parent=  3  combined= 15
astraagro        group= 33  parent=  2  combined= 34
genting          group= 14  parent= 10  combined= 15
sipef            group= 11  parent=  2  combined= 11
firstresources   group= 15  parent=  1  combined= 16


## Pre-meeting diagnostics

Two checks the supervisor is likely to probe.

In [20]:
# --- Pre-meeting diagnostics ---

# 1. Did any company return zero mills? Those numbers are not usable.
print("=== MILL COUNTS ===")
for _c in COMPANIES:
    _n = len(mills_master[mills_master["company"] == _c])
    _flag = "   <-- ZERO, keyword needs fixing" if _n == 0 else ""
    print(f"  {_c:16s} {_n:3d} mills{_flag}")

# 2. Do any Wilmar catchments overlap? (open limitation - double counting)
from itertools import combinations
import math

_w = mills_master[mills_master["company"] == "wilmar"]
_close = sum(
    1 for (_, a), (_, b) in combinations(_w.iterrows(), 2)
    if math.dist((a["latitude"], a["longitude"]), (b["latitude"], b["longitude"])) * 111 < 20
)
print(f"\n=== OVERLAP ===")
print(f"Wilmar mill pairs closer than 20 km: {_close}")
print("(each such pair shares catchment area, so some hectares are counted twice)")

=== MILL COUNTS ===
  wilmar            45 mills
  gar               50 mills
  sdguthrie         42 mills
  musimmas          18 mills
  ioi               15 mills
  klk               30 mills
  bumitama          14 mills
  astraagro         34 mills
  genting           15 mills
  sipef             11 mills
  firstresources    16 mills

=== OVERLAP ===
Wilmar mill pairs closer than 20 km: 19
(each such pair shares catchment area, so some hectares are counted twice)


## Single-mill detail

Year-by-year breakdown and the EUDR before/after split.

In [21]:
# --- Year-by-year loss for one mill (Wilmar Sabahmas, Sabah) ---

def loss_by_year(lon, lat, buffer_m=10000):
    region = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    out = {}
    for yr in range(21, 26):
        img = hansen.select("lossyear").eq(yr).multiply(pixel_ha)
        v = img.reduceRegion(ee.Reducer.sum(), region, 30, maxPixels=1e10).getInfo()
        out[2000 + yr] = round(v.get("lossyear") or 0)
    return out


def before_after(lon, lat, buffer_m=10000):
    region = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    _pre = hansen.select("lossyear").gte(1).And(hansen.select("lossyear").lte(20))
    _post = hansen.select("lossyear").gte(21)
    b = _pre.multiply(pixel_ha).reduceRegion(ee.Reducer.sum(), region, 30, maxPixels=1e10).getInfo()
    a = _post.multiply(pixel_ha).reduceRegion(ee.Reducer.sum(), region, 30, maxPixels=1e10).getInfo()
    return round(b.get("lossyear") or 0), round(a.get("lossyear") or 0)


_lon, _lat = 118.405246, 5.179162

print("Year-by-year loss (ha):")
for _y, _v in loss_by_year(_lon, _lat).items():
    print(f"  {_y}: {_v:,}")

_pre_ha, _post_ha = before_after(_lon, _lat)
print(f"\nEUDR split - 2001-2020: {_pre_ha:,} ha  |  2021-2025: {_post_ha:,} ha")

Year-by-year loss (ha):
  2021: 2,025
  2022: 589
  2023: 1,438
  2024: 1,372
  2025: 1,251

EUDR split - 2001-2020: 16,488 ha  |  2021-2025: 6,675 ha


## Master table

This is the CSV every number in the thesis must trace back to.

In [22]:
master = (
    mills_master
    .groupby("company", as_index=False)
    .agg(
        n_mills=("mill_name", "count"),
        forest_2000_ha=("forest_2000_ha", "sum"),
        loss_total_ha=("loss_total_ha", "sum"),
        loss_post2020_ha=("loss_post2020_ha", "sum"),
    ))
master["loss_pct_of_forest"] = (
    master["loss_post2020_ha"] / master["forest_2000_ha"] * 100).round(2)

# Normalises for company size — answers "is SD Guthrie top only because it has more mills?"
master["ha_per_mill"] = (master["loss_post2020_ha"] / master["n_mills"]).round(0)
master = master.sort_values("loss_post2020_ha", ascending=False).reset_index(drop=True)
master.to_csv("data/mills/master_forest_loss.csv", index=False)
master

,company,n_mills,forest_2000_ha,loss_total_ha,loss_post2020_ha,loss_pct_of_forest,ha_per_mill
0,gar,50,1.234269e+06,792970.995367,171236.168128,13.87,3425.0
1,sdguthrie,42,9.863202e+05,700217.038743,142131.279201,14.41,3384.0
2,wilmar,45,1.119815e+06,711440.929911,127605.651214,11.40,2836.0
3,klk,30,7.571609e+05,502073.696143,92208.918345,12.18,3074.0
4,astraagro,34,8.103222e+05,507344.985693,87198.086458,10.76,2565.0
5,ioi,15,3.906211e+05,292261.771873,74385.364496,19.04,4959.0
6,musimmas,18,4.635893e+05,301860.767177,53059.552603,11.45,2948.0
7,genting,15,3.821293e+05,243363.718735,44362.418926,11.61,2957.0
8,firstresources,16,3.897264e+05,267096.286626,40090.269846,10.29,2506.0
9,bumitama,14,3.529216e+05,237175.975455,30549.328832,8.66,2182.0


## Map visualization (SD Guthrie example)
Interactive forest cover/loss map, `mill_coords` currently holds SD Guthrie mills. Rerun `get_company_mills(uml, keyword, name)` first to visualize another company.

In [23]:
gfc_map = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")

# Ek picture banao: hara = jungle 2000 mein, laal = 2020 ke baad kata
image = gfc_map.select("treecover2000").gt(30).multiply(1).add(
        gfc_map.select("lossyear").gte(21).multiply(2))

# Sab mills ke GPS coordinates ek interactive map par dikhao
center_lat = mill_coords["latitude"].mean()
center_lon = mill_coords["longitude"].mean()

mills_map = folium.Map(location=[center_lat, center_lon], zoom_start=8)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google Satellite",
    name="Google Satellite",
    overlay=False,
).add_to(mills_map)

map_id_dict = image.getMapId(
    {"min": 0, "max": 3, "palette": ["white", "green", "green", "red"]}
)
folium.raster_layers.TileLayer(
    tiles=map_id_dict["tile_fetcher"].url_format,
    attr="Google Earth Engine",
    name="Forest cover / loss",
    overlay=True,
    control=True,
).add_to(mills_map)

for _, _row in mill_coords.iterrows():
    folium.Marker(
        location=[_row["latitude"], _row["longitude"]],
        popup=f"{_row['Mill Name']} ({_row['latitude']}, {_row['longitude']})",
    ).add_to(mills_map)

folium.LayerControl().add_to(mills_map)
mills_map

## Sensitivity check (Week 2 task)

Recompute at 5 km and 15 km buffers. If the company ranking holds, that is a
robustness finding worth a paragraph in Results. If it shifts, that is a
limitation worth a paragraph. Either outcome is useful — run it and report it.

In [24]:
# Sensitivity: does the ranking survive a different buffer radius?
# Start with 2-3 companies to gauge how long the full run would take.

def sensitivity_check(uml_df, companies, radii=(5000, 10000, 15000)):
    out = []
    for _name, _keyword in companies.items():
        for _r in radii:
            _df = company_forest_loss(uml_df, _keyword, f"{_name}_{_r//1000}km", buffer_m=_r)
            out.append({
                "company": _name,
                "buffer_km": _r // 1000,
                "loss_post2020_ha": _df["loss_post2020_ha"].sum(),
            })
    return pd.DataFrame(out)

# subset = {k: COMPANIES[k] for k in ["wilmar", "gar", "sdguthrie"]}
# sens = sensitivity_check(uml, subset)
# sens.pivot(index="company", columns="buffer_km", values="loss_post2020_ha")

In [25]:
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

COMPANIES = {
    "wilmar":         "WILMAR",
    "gar":            "GOLDEN AGRI|SINAR MAS",
    "sdguthrie":      "SIME DARBY",
    "musimmas":       "MUSIM MAS",
    "ioi":            "IOI CORPORATION|IOI GROUP",
    "klk":            "KUALA LUMPUR KEPONG|KLK",
    "bumitama":       "BUMITAMA",
    "astraagro":      "ASTRA AGRO|ASTRA",
    "genting":        "GENTING",
    "sipef":          "SIPEF",
    "firstresources": "FIRST RESOURCES",
    "apical":         "APICAL",
}

all_mills = []

for company, keyword in COMPANIES.items():
    mask = (
        uml[group_col].str.contains(keyword, case=False, na=False, regex=True) |
        uml["Parent Company"].str.contains(keyword, case=False, na=False, regex=True)
    )
    mills = uml[mask].drop_duplicates(subset=["Mill Name", "GPS coordinates"]).copy()

    # Safe coordinate parsing
    def parse_coord(series, idx):
        try:
            parts = series.str.split(",", expand=True)
            if idx in parts.columns:
                return pd.to_numeric(parts[idx].str.strip(), errors="coerce")
        except:
            pass
        return pd.Series([None]*len(series), index=series.index)

    mills["latitude"]  = parse_coord(mills["GPS coordinates"], 0)
    mills["longitude"] = parse_coord(mills["GPS coordinates"], 1)
    mills = mills.dropna(subset=["latitude", "longitude"])

    mills["company"] = company
    all_mills.append(mills[[
        "company", "Mill Name", group_col, "Parent Company",
        "Country", "latitude", "longitude", "RSPO Status"
    ]])

    print(f"{company:16s} {len(mills):3d} mills")

final = pd.concat(all_mills, ignore_index=True)
final.columns = ["company", "mill_name", "group_name", "parent_company",
                 "country", "latitude", "longitude", "rspo_status"]

final.to_csv("data/all_company_mills.csv", index=False)

print(f"\nTotal mills: {len(final)}")
print("Saved!")
final

wilmar            45 mills
gar               50 mills
sdguthrie         41 mills
musimmas          18 mills
ioi               15 mills
klk               30 mills
bumitama          14 mills
astraagro         34 mills
genting           15 mills
sipef             11 mills
firstresources    16 mills
apical             0 mills

Total mills: 289
Saved!


,company,mill_name,group_name,parent_company,country,latitude,longitude,rspo_status
0,wilmar,SABAHMAS,WILMAR GROUP,SABAHMAS PLANTATION SDN BHD,Malaysia,5.179162,118.405246,RSPO Certified
1,wilmar,SAREMAS 1,WILMAR GROUP,SAREMAS SDN BHD,Malaysia,3.524921,113.744412,RSPO Certified
2,wilmar,SAREMAS 2,WILMAR GROUP,SAREMAS SDN BHD,Malaysia,3.450453,113.765561,RSPO Certified
3,wilmar,TERUSAN,WILMAR GROUP,SAPI PLANTATION SDN BHD,Malaysia,5.831514,117.340459,RSPO Certified
4,wilmar,SAPI,WILMAR GROUP,SAPI PLANTATION SDN BHD,Malaysia,5.733519,117.387404,RSPO Certified
...,...,...,...,...,...,...,...,...
284,firstresources,LIMPAH SEJAHTERA,FIRST RESOURCES,LIMPAH SEJAHTERA,Indonesia,-1.716225,110.354369,Not RSPO Certified
285,firstresources,PERDANA INTI SAWIT PERKASA 2,FIRST RESOURCES,PERDANA INTISAWIT PERKASA,Indonesia,1.167333,100.704778,Not RSPO Certified
286,firstresources,KETAPANG AGRO LESTARI,FIRST RESOURCES,KETAPANG AGRO LESTARI,Indonesia,-0.875633,115.891653,Not RSPO Certified
287,firstresources,UMEKA SARI PRATAMA,FIRST RESOURCES,UMEKAH SARI PRATAMA,Indonesia,-2.1238,110.9809,Not RSPO Certified


In [26]:
import pandas as pd
mills_df = pd.read_csv("data/all_company_mills.csv")
gar = mills_df[mills_df["company"] == "gar"][["mill_name", "latitude", "longitude"]].dropna()

for _, row in gar.iterrows():
    print(f'  {{name:"{row["mill_name"]}", lon:{row["longitude"]:.6f}, lat:{row["latitude"]:.6f}}},')

  {name:"UJUNG TANJUNG", lon:101.260750, lat:0.969733},
  {name:"LIBO", lon:101.206389, lat:0.928611},
  {name:"SAM-SAM", lon:101.300200, lat:0.937500},
  {name:"SEKIJANG", lon:101.044283, lat:0.833933},
  {name:"NAGA SAKTI", lon:101.049567, lat:0.782700},
  {name:"RAMARAMA", lon:101.076381, lat:0.533494},
  {name:"BUMI PALMA", lon:102.983333, lat:-0.598056},
  {name:"INDRA SAKTI", lon:102.305851, lat:-0.569200},
  {name:"PADANG HALABAN", lon:99.839444, lat:2.319167},
  {name:"LANGGA PAYUNG PALM OIL MILL", lon:99.886944, lat:1.654444},
  {name:"BATU AMPAR", lon:116.021900, lat:-3.198100},
  {name:"TANAH LAUR", lon:115.283056, lat:-3.790278},
  {name:"HANAU", lon:112.109694, lat:-2.360861},
  {name:"SUNGAI RUNGAU", lon:112.334000, lat:-2.320600},
  {name:"SEMILAR", lon:112.340417, lat:-2.248111},
  {name:"SUNGAI BUAYA", lon:105.438839, lat:-4.128611},
  {name:"SUNGAI MERAH", lon:105.588769, lat:-4.218444},
  {name:"JELATANG", lon:102.485833, lat:-2.072222},
  {name:"LANGLING", lon:102.3

In [29]:
import pandas as pd
import json

master = pd.read_csv("data/mills/master_forest_loss.csv")

# Company name mapping — exact names
name_map = {
    "gar": "GAR",
    "sdguthrie": "SD Guthrie",
    "wilmar": "Wilmar",
    "klk": "KLK",
    "astraagro": "Astra Agro",
    "ioi": "IOI",
    "musimmas": "Musim Mas",
    "genting": "Genting",
    "firstresources": "First Resources",
    "bumitama": "Bumitama",
    "sipef": "SIPEF",
}

js_rows = []
for _, r in master.iterrows():
    company_key = r["company"].lower().replace(" ", "")
    js_rows.append({
        "company": name_map.get(r["company"], r["company"].title()),
        "n_mills": int(r["n_mills"]),
        "forest_2000": int(r["forest_2000_ha"]),
        "loss_total": int(r["loss_total_ha"]),
        "loss_post2020": int(r["loss_post2020_ha"]),
        "pct": round(float(r["loss_pct_of_forest"]), 2),
        "ha_per_mill": int(r["ha_per_mill"])
    })

js_output = "const data = " + json.dumps(js_rows, indent=2) + ";"

with open("data/data_array.js", "w") as f:
    f.write(js_output)

# Verify
print("Companies in JS array:")
for row in js_rows:
    print(f"  {row['company']:15s} | mills:{row['n_mills']:3d} | post2020:{row['loss_post2020']:,} ha | {row['pct']}%")

Companies in JS array:
  GAR             | mills: 50 | post2020:171,236 ha | 13.87%
  SD Guthrie      | mills: 42 | post2020:142,131 ha | 14.41%
  Wilmar          | mills: 45 | post2020:127,605 ha | 11.4%
  KLK             | mills: 30 | post2020:92,208 ha | 12.18%
  Astra Agro      | mills: 34 | post2020:87,198 ha | 10.76%
  IOI             | mills: 15 | post2020:74,385 ha | 19.04%
  Musim Mas       | mills: 18 | post2020:53,059 ha | 11.45%
  Genting         | mills: 15 | post2020:44,362 ha | 11.61%
  First Resources | mills: 16 | post2020:40,090 ha | 10.29%
  Bumitama        | mills: 14 | post2020:30,549 ha | 8.66%
  SIPEF           | mills: 11 | post2020:27,340 ha | 10.42%
